### 大量データを処理するワークフロー

定量分析（Quantitative Analysis）は、データや事象を数値として捉え、統計的・数学的な手法で分析を行う方法です。定量分析は「数値的なデータ」に基づくため、客観的で正確な判断に適しています。

：pandas が適している。

定量分析の手法：

定量分析には、以下のような手法があります。

基本統計量の算出：平均値、中央値、分散、標準偏差など
相関分析：データ同士の相関関係を数値化
回帰分析：複数の変数の関係性から将来の予測を立てる
仮説検定：サンプルデータをもとに、特定の仮説が正しいかを検証
可視化：グラフや表を使ったデータの視覚化

定性分析とは
定性分析（Qualitative Analysis）は、データや事象を数値ではなく、文章やカテゴリ、事例のような「質的な情報」として捉え、意味やパターンを解釈する方法です。定性分析は「意味的なデータ」や「人の意見」を扱うため、解釈や洞察が求められます。

：言語モデルが適している。

定性分析の手法：

定性分析には、以下のような手法があります。

テキストマイニング: 大量の文章データからテーマやパターンを抽出
感情分析: レビューやコメントからポジティブ/ネガティブの傾向を分析
テーマ分析: データの中から共通のテーマや概念を見つけ出す
コーディング: テキストにタグやコードを付与し、特定のカテゴリごとに分析

組み合わせることで精度の高い分析が可能に。

In [1]:
#　売上データから販売戦略をつくる


# モジュールのインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

データの読み込みと前処理を行う関数

In [2]:
# データの読み込みと前処理を行う関数

def load_and_process_data(file_path):
    # Excelファイルを読み込む
    df = pd.read_excel(file_path, sheet_name='売上データ')

    # 「売上」列を追加（単価 * 数量）
    df["売上"] = df["単価"] * df["数量"]

    # 「売上日」列をdatetime型に変換
    df["売上日"] = pd.to_datetime(df["売上日"])

    # 「年月」列を追加（売上日の年と月を抽出）
    df["年月"] = df["売上日"].dt.strftime('%Y-%m')

    # データフレームを返す
    return df

集計を行う関数

＊渡されたデータフレームに基づいて、月ごとのカテゴリー別売上合計を計算し、結果をピボットテーブルとして返す関数

In [3]:
# 集計を行う関数
def sum_sales_data(df):
    # 月ごとのカテゴリー別売上合計を集計
    pivot_table = pd.pivot_table(
        df,
        index='カテゴリー',   # 行にする項目
        columns='年月',       # 列にする項目
        values='売上',        # 集計対象の項目
        aggfunc='sum',        # 集計方法（ここでは合計を指定）
        fill_value=0          # NaNの代わりに埋める値
    )

    # ピボットテーブルを返す
    return pivot_table

データをプロンプトに変換する関数

与えられた売上データ（df）と集計結果（pivot_table）を使って、OpenAI API へのプロンプトを返却する


In [4]:
# データをプロンプトに変換する関数
def convert_to_prompt(df, pivot_table):
    # データフレーム全体を文字列に変換
    sales_data_text = df.astype(str)

    # 集計結果を文字列に変換
    pivot_table_text = pivot_table.astype(str)

    # プロンプトの作成
    prompt_text = f"""
    売上データ:
    {sales_data_text}
    月ごとのカテゴリー別売上合計：
    {pivot_table_text}
    上記の「売上データ」と「月ごとのカテゴリー別売上合計」をもとに、
    カテゴリー毎の売上戦略を考案してください。
    """

    # プロンプトを返す
    return prompt_text

こんなプロンプトが出来上がる。すごい。

売上データ:
カテゴリー 売上日 単価 数量 売上
食品 2024-01-01 100 2 200
食品 2024-02-01 120 3 360
家電 2024-01-15 200 1 200
家電 2024-02-10 250 2 500

月ごとのカテゴリー別売上合計：
カテゴリー 2024-01 2024-02
食品 200 360
家電 200 500

上記の「売上データ」と「月ごとのカテゴリー別売上合計」を・・・

OpenAI APIの呼び出しを行って、結果をかえす関数

In [5]:
# OpenAI APIの呼び出しを行う関数
def get_openai_response(
        client, 
        prompt_text, 
        model_name="gpt-4o-mini"
        ):
    # 役割を設定
    role = (f"あなたはマーケティング分野に精通したデータサイエンティスト"
            "です。企業の成長をサポートするために、効果的なインサイトを"
            "提供します。")

    # APIへリクエスト
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": role},
            {"role": "user", "content": prompt_text},
        ],
    )

    # 結果を取得
    result = response.choices[0].message.content.strip()

    # 結果を返す
    return result

resultに受け取った結果を、ファイルに保存する関数。

言語モデルからの出力はmarkdown形式がほとんど。ファイルはmd形式とする

In [6]:
# 結果をファイルに保存する関数
def save_result_to_file(result, file_path="カテゴリー毎の売上戦略.md"):
    with open(file_path, mode="w", encoding="utf-8") as file:
        file.write(result)

関数を作り切ったら、メインとなるワークフローをつくる。

各段階で、プリントして状況がわかるように。


In [7]:
# ワークフロー（メイン処理）
def main():
    print("処理を開始します。")

    # 1.データの読み込みと前処理
    print("（1/5）データの読み込みと前処理")
    df = load_and_process_data('サンプルデータ.xlsx')

    # 2.集計
    print("（2/5）集計")
    pivot_table = sum_sales_data(df)

    # 3.プロンプト生成
    print("（3/5）プロンプト生成")
    prompt_text = convert_to_prompt(df, pivot_table)

    # 4.OpenAI APIからの応答を取得
    print("（4/5）OpenAI APIからの応答を取得")
    result = get_openai_response(client, prompt_text)

    # 5.結果をファイルに保存
    print("（5/5）結果をファイルに保存")
    save_result_to_file(result)
    print("分析結果を保存しました。")

ここまでで準備完了。
あとはメインフローの　main()関数を実行

In [8]:
# ワークフローの実行。
main()

処理を開始します。
（1/5）データの読み込みと前処理
（2/5）集計
（3/5）プロンプト生成
（4/5）OpenAI APIからの応答を取得
（5/5）結果をファイルに保存
分析結果を保存しました。
